In [63]:
import json
import re
import sqlite3
import pandas as pd

In [64]:
db = sqlite3.connect("/content/download (4)")

In [65]:
pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    db
)

,name
0,members
1,books
2,checkouts


In [66]:
pd.read_sql_query("SELECT * from members",db)

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05
1,1002,Fares,Saleh,9.0,Maadi,Active,None
2,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23
3,1004,Fares,Wahba,7.0,Maadi,inactive,2024-10-09
4,1005,Youssef,Halim,9.0,Maadi,Active,2024-05-05
...,...,...,...,...,...,...,...
75,1076,Dina,Wahba,7.0,Shubra,Active,2025-12-13
76,1077,Lina,Rashad,6.0,Shubra,Active,2023-07-08
77,1078,Habiba,Osman,7.0,Shubra,INACTIVE,2023-08-05
78,1079,Rana,Osman,8.0,Shubra,Active,2024-10-27


In [67]:
pd.read_sql_query("SELECT * from checkouts",db)


,checkout_id,member_id,book_id,checkout_date,return_date
0,9263,1047,517,2024-10-21,2024-11-07
1,9340,1072,513,2025-08-24,2025-09-01
2,9231,1053,523,2024-02-04,2024-02-16
3,9129,1032,513,2025-06-21,2025-06-29
4,9370,1079,511,2025-11-11,2025-12-03
...,...,...,...,...,...
386,9232,1044,513,2025-05-26,2025-06-11
387,9084,1008,511,2024-06-27,2024-07-22
388,9116,1024,519,2025-01-10,2025-02-09
389,9352,1076,501,2025-02-02,None


In [68]:
pd.read_sql_query("SELECT * from books",db)


,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez
5,506,The Paper Boat Club,Aya Hafez
6,507,Fossils and Fireflies,Dalia Serry
7,508,The Quiet Observatory,Dalia Serry
8,509,Marbles and Mirrors,Diaa Sultan
9,510,The Missing Metronome,Diaa Sultan


In [69]:
query1 = '''
SELECT members.first_name AS member,
       count(checkouts.book_id) AS BookCount

from checkouts

join members on members.member_id = checkouts.member_id

GROUP BY members.member_id, members.first_name

'''


In [70]:
df1 = pd.read_sql_query(query1, db)
df1

,member,BookCount
0,Salma,1
1,Fares,2
2,Bassel,9
3,Youssef,3
4,Layla,1
...,...,...
57,Malak,9
58,Dina,7
59,Lina,6
60,Rana,10


In [71]:
query2 = '''
SELECT title ,author  FROM books
WHERE author like 'A%'
'''

In [72]:
df2 = pd.read_sql_query(query2, db)
df2

,title,author
0,The Silver Kite,Amina Darwish
1,Desert Compass,Amina Darwish
2,The Lantern Maker,Adel Roushdy
3,Rooftop Astronomers,Adel Roushdy
4,Letters to the Nile,Aya Hafez
5,The Paper Boat Club,Aya Hafez


In [73]:
query3 = '''
SELECT books.title AS book,
       count(checkouts.book_id) AS BookCount

from checkouts

join books on books.book_id = checkouts.book_id

GROUP BY books.book_id,books.title

order by BookCount DESC
LIMIT 5

'''

In [74]:
df3 = pd.read_sql_query(query3, db)
df3

,book,BookCount
0,The Silver Kite,57
1,Fossils and Fireflies,55
2,Circuits for Beginners,46
3,Kites Over Cairo,38
4,Storms and Sailboats,25


In [75]:
query4 = '''
SELECT members.first_name AS member,
       count(checkouts.book_id) AS borrowing

from checkouts

join members on members.member_id = checkouts.member_id

GROUP BY members.member_id, members.first_name
order by borrowing DESC
LIMIT 10
'''


In [76]:
df4 = pd.read_sql_query(query4, db)
df4

,member,borrowing
0,Aya,25
1,Sherif,21
2,Ziad,19
3,Nour,18
4,Mostafa,18
5,Ahmed,17
6,Youssef,17
7,Adam,17
8,Reem,16
9,Sara,16


In [77]:
query5 = '''
SELECT
    members.first_name AS name,
    COUNT(checkouts.book_id) AS BookCount
FROM checkouts
JOIN members ON members.member_id = checkouts.member_id
WHERE members.neighborhood = 'Maadi'
GROUP BY members.member_id, members.first_name
ORDER BY MAX(checkouts.checkout_date) DESC
LIMIT 10 OFFSET 10;
'''

In [78]:
df5 = pd.read_sql_query(query5, db)
df5

,name,BookCount
0,Dina,7
1,Youssef,6
2,Mostafa,3
3,Layla,1
4,Sherif,3
5,Salma,1
6,Fares,2
7,Habiba,1


In [79]:
conn = sqlite3.connect("/content/download (4)")
members = pd.read_sql_query("SELECT * FROM members", conn)
books = pd.read_sql_query("SELECT * FROM books", conn)
checkouts = pd.read_sql_query("SELECT * FROM checkouts", conn)
conn.close()

data = checkouts.merge(members, on="member_id", how="left")

print("Checkouts per member:")
print(data.groupby("member_id").size().head())

Checkouts per member:
member_id
1001    1
1002    2
1003    9
1005    3
1006    1
dtype: int64


In [80]:
with open("/content/download (1)", encoding="utf-8") as f:
    book_details = pd.DataFrame(json.load(f))

data = data.merge(books, on="book_id", how="left")
data = data.merge(book_details, on="book_id", how="left")
data["source"] = "database"

In [81]:
with open("/content/download (2)", encoding="utf-8") as f:
    html = f.read()
rows = re.findall(r"<tr><td>(\d+)</td><td>(\d+)</td><td>([\d-]+)</td></tr>", html)

kickoff = pd.DataFrame(rows, columns=["member_id", "book_id", "checkout_date"])
kickoff["member_id"] = kickoff["member_id"].astype(int)
kickoff["book_id"] = kickoff["book_id"].astype(int)
kickoff["checkout_id"] = ["RK-" + str(i + 1) for i in range(len(kickoff))]
kickoff["return_date"] = None  # not recorded on the page
kickoff["source"] = "reading_kickoff"

kickoff = kickoff.merge(members, on="member_id", how="left")
kickoff = kickoff.merge(books, on="book_id", how="left")
kickoff = kickoff.merge(book_details, on="book_id", how="left")

data = pd.concat([data, kickoff], ignore_index=True)

In [82]:
print("\nFinal dataset:")
print("Records:", len(data))
print("Columns:", list(data.columns))
print(data.sample(5))

data.to_csv("task1_combained_data.csv", index=False)
print("\nSaved combined_checkouts.csv")


Final dataset:
Records: 417
Columns: ['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date', 'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status', 'join_date', 'title', 'author', 'genre', 'pages', 'publication_year', 'publisher', 'source']
    checkout_id  member_id  book_id checkout_date return_date first_name  \
391        RK-1       1026      522    2025-07-11        None       Nada   
26         9256       1059      521    2025-07-09        None       Lina   
191        9080       1008      501    2024-03-20  2024-04-18       Ziad   
197        9216       1050      519    2024-08-07        None      Fares   
53         9056       1010      519    2025-07-11  2025-07-27       Nour   

    last_name  grade neighborhood membership_status   join_date  \
391     Saleh    7.0    Nasr City          inactive  2023-11-16   
26      Halim    7.0   Heliopolis            Active  2024-01-06   
191     Saleh    8.0        Maadi            active  2025-10-28   

In [83]:
df = pd.read_csv("/content/task1_combained_data.csv")

In [84]:
print("Starting rows:", len(df))

Starting rows: 417


In [85]:
print("\nMissing values per column (before cleaning):")
print(df.isna().sum()[df.isna().sum() > 0])


Missing values per column (before cleaning):
return_date          91
first_name            5
last_name             5
grade                41
neighborhood          5
membership_status     5
join_date            11
publication_year     35
dtype: int64


In [86]:
mode_return_date = df["return_date"].mode()[0]
df["return_date"] = df["return_date"].fillna(mode_return_date)
print(f"\nreturn_date: filled with the most common value ({mode_return_date})")


return_date: filled with the most common value (2025-01-11)


In [87]:
mean_grade = round(df["grade"].mean())
df["grade"] = df["grade"].fillna(mean_grade)
print(f"grade: filled with the average grade ({mean_grade})")

grade: filled with the average grade (8)


In [88]:
mode_join_date = df["join_date"].mode()[0]
df["join_date"] = df["join_date"].fillna(mode_join_date)
print(f"join_date: filled with the most common value ({mode_join_date})")

join_date: filled with the most common value (2025-01-22)


In [89]:
df["publication_year"] = df["publication_year"].fillna("Unknown")
print("publication_year: filled with \"Unknown\"")

publication_year: filled with "Unknown"


In [90]:
before = len(df)
df = df.drop_duplicates()
print(f"\nRemoved {before - len(df)} true duplicate row(s)")


Removed 8 true duplicate row(s)


In [91]:
for col in ["neighborhood", "membership_status"]:
    df[col] = df[col].str.strip().str.title()

In [92]:
conn = sqlite3.connect("/content/download (4)")
members = pd.read_sql_query("SELECT member_id FROM members", conn)
conn.close()

before = len(df)
df = df[df["member_id"].isin(members["member_id"])]
print(f"Removed {before - len(df)} row(s) with a member_id not in the members table")

Removed 5 row(s) with a member_id not in the members table


In [93]:
print("\nFinal cleaned dataset:")
print("Records:", len(df))
print("Columns:", list(df.columns))
print(df.sample(5))

df.to_csv("task2_cleaned_data.csv", index=False)
print("\nSaved task2_cleaned_data.csv")


Final cleaned dataset:
Records: 404
Columns: ['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date', 'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status', 'join_date', 'title', 'author', 'genre', 'pages', 'publication_year', 'publisher', 'source']
    checkout_id  member_id  book_id checkout_date return_date first_name  \
256        9320       1073      501    2025-07-19  2025-08-13     Yassin   
172        9105       1027      501    2025-03-05  2025-03-25    Mostafa   
55         9140       1034      513    2025-03-06  2025-04-02        Aya   
364        9221       1055      520    2025-04-12  2025-04-18    Mostafa   
218        9125       1039      501    2025-06-17  2025-01-11       Nada   

    last_name  grade neighborhood membership_status   join_date  \
256     Kamel    7.0      Zamalek          Inactive  2025-08-26   
172     Fouad    9.0    Nasr City            Active  2023-09-25   
55      Wahba    9.0    Nasr City            Active  2025-